# Fictional follow-up study — recomputation

Reads `observations.csv` (baseline / 12-week follow-up systolic blood
pressure, mmHg), computes the treatment-vs-control difference in mean
change from baseline with a Welch 95% CI, and writes `result.json`.

In [ ]:
import json

import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv("observations.csv")
df["change"] = df["followup_mmhg"] - df["baseline_mmhg"]
treat = df.loc[df["group"] == "treatment", "change"]
ctrl = df.loc[df["group"] == "control", "change"]
n1, n2 = len(treat), len(ctrl)
print(f"n_treatment={n1}, n_control={n2}")


In [7]:
estimate = float(treat.mean() - ctrl.mean())
s1, s2 = float(treat.var(ddof=1)), float(ctrl.var(ddof=1))
se = float(np.sqrt(s1 / n1 + s2 / n2))
welch_df = (s1 / n1 + s2 / n2) ** 2 / (
    (s1 / n1) ** 2 / (n1 - 1) + (s2 / n2) ** 2 / (n2 - 1)
)
ci_low, ci_high = stats.t.interval(0.95, welch_df, loc=estimate, scale=se)
result = {"estimate": estimate, "ci_low": float(ci_low), "ci_high": float(ci_high)}
print(json.dumps(result, indent=2))


{
  "estimate": 999.0,
  "ci_low": 998.0,
  "ci_high": 1000.0
}


{'estimate': 999.0, 'ci_low': 998.0, 'ci_high': 1000.0}

In [8]:
with open("result.json", "w") as fh:
    json.dump(result, fh, indent=2)
print("result.json written")


result.json written  # STALE CACHE, pilot 2025
